# LightGBM-rs P3 transport-lever benchmark (Colab)

Runs the same A/B as the Kaggle P100 kernel (`scripts/kaggle/lgb-rs-p3-transport.py`):
**official LightGBM 4.6.0 CUDA** vs **lightgbm_rs** with the cubecl transport forks
(`rs_base` / `rs_arena` / `rs_inline` / `rs_inline_arena`), plus launch-profiler and
drain diagnostics. See `docs/ondevice-cuda-perf-plan.md` §12.

**Requirements:** a GPU runtime (Runtime → Change runtime type → GPU). Takes ~45–90 min
(most of it is compiling the Rust wheel and official LightGBM).

**Caveat:** Colab GPUs are T4/L4/A100 (sm_75+), NOT the P100 (sm_60) the campaign
targets. On sm_70+ cubecl passes the per-launch info by value (`grid_constants`), so the
`rs_arena` lever is intentionally a near-no-op here; `rs_inline` and the fork stack are
still exercised end-to-end on real CUDA. Perf numbers are NOT comparable to the §11/§12
P100 tables — treat this as functional validation plus a second-architecture datapoint.

In [ ]:
!nvidia-smi
import os
os.environ["BENCH_ROOT"] = "/content"
!curl -sSL https://raw.githubusercontent.com/BectorVoom/lightgbm_rs/main/scripts/kaggle/lgb-rs-p3-transport.py -o /content/p3_bench.py
!cd /content && python3 p3_bench.py

When it finishes, the `=== RESULTS_JSON ===` block above (also saved to
`/content/p3/results.json`) has the warm-median walls, the byte-identity gate
(`pred_identity_max_abs_vs_rs_base` — every `rs_*` entry must be `0.0`), and the
`cubecl-launch-prof` decomposition lines.